# Adding spin projections

:::{autolink-concat}
:::

The default workflow solves reactions at the level of spin **magnitudes**: it does not track the spin projections of the states, because those only multiply the number of transitions without affecting which intermediate states are allowed. The underlying constraint solver, however, is agnostic about which quantum numbers it propagates — it solves for whatever **facts**, **domains**, and **rules** the {class}`.QNProblemSet`s declare. This page demonstrates that extensibility by re-introducing spin projections for the reaction $J/\psi \to \gamma f_2(1270) \to \gamma\pi^0\pi^0$ and pruning the helicity combinations with {func}`.helicity_conservation`.

:::{seealso}
The level of detail can be reduced further as well: with `ls_couplings=False`, {func}`.generate_transitions` and {func}`.create_qn_problem_sets` do not enumerate $LS$-combinations either. The existence rules {class}`.SpinCoupling` and {class}`.SpinParityCoupling` then only check that *some* coupling with $L \leq L_\mathrm{max}$ can produce the spins and parities, and the allowed $(L, S)$ combinations can be reconstructed from the solutions afterwards (see {func}`.create_interaction_settings`).
:::

In [ ]:
from collections import defaultdict
from fractions import Fraction

import attrs
import graphviz

import qrules
import qrules.io
from qrules.conservation_rules import helicity_conservation, spin_validity
from qrules.quantum_numbers import EdgeQuantumNumbers
from qrules.settings import CONSERVATION_LAW_PRIORITIES, EDGE_RULE_PRIORITIES
from qrules.solving import QNProblemSet
from qrules.topology import MutableTransition
from qrules.workflow import create_qn_problem_sets, find_qn_transitions

PDG = qrules.load_pdg()

## A projection-free reaction

First, create the default problem sets with {func}`.create_qn_problem_sets` (see {doc}`/usage/qn-transitions`). The final-state grouping selects the decay topology in which the two pions form the $f_2(1270)$ resonance:

In [ ]:
collection = create_qn_problem_sets(
    initial_state=["J/psi(1S)"],
    final_state=["gamma", "pi0", "pi0"],
    particle_db=PDG,
    allowed_intermediate_particles=["f(2)(1270)"],
    allowed_interaction_types=["strong", "EM"],
    max_angular_momentum=2,
    final_state_groupings=[[["pi0", "pi0"]]],
)
sum(len(problem_sets) for problem_sets in collection.problem_sets.values())

Each {class}`.QNProblemSet` declares the known quantum numbers of the initial and final states (its **facts**, {attr}`~.QNProblemSet.initial_facts`) and the value ranges to solve for on the intermediate edges and interaction nodes (its **domains**, part of the {attr}`~.QNProblemSet.solving_settings`). Neither mentions a {attr}`~.EdgeQuantumNumbers.spin_projection`:

In [ ]:
sorted({
    qn_type.__name__
    for problem_sets in collection.problem_sets.values()
    for problem_set in problem_sets
    for prop_map in problem_set.initial_facts.states.values()
    for qn_type in prop_map
})

## Adding facts and domains

Spin projections enter the problem sets through the same two channels as any other quantum number. The external edges get {attr}`~.EdgeQuantumNumbers.spin_projection` **facts**: a single value fixes the projection, while a `list` of values is registered by the {class}`.CSPSolver` as a variable with that list as its domain, so that several projection cases are solved within a single problem set. Here, the $J/\psi$ is restricted to $\lambda=\pm1$ (as it is produced from $e^+e^-$ collisions), the photon to its two helicity states, and the pions are fixed to $0$:

In [ ]:
spin_projections = {
    "J/psi(1S)": [Fraction(-1), Fraction(+1)],
    "gamma": [Fraction(-1), Fraction(+1)],
    "pi0": Fraction(0),
}

The intermediate edge gets a projection **domain**, derived from the spin magnitudes that its {class}`.EdgeSettings` allow, together with the {func}`.spin_validity` rule, which requires $|m| \leq s$ on a single edge. Every rule carries a priority (higher priorities are evaluated first); take the defaults from {obj}`.EDGE_RULE_PRIORITIES` and {obj}`.CONSERVATION_LAW_PRIORITIES`:

In [ ]:
def add_spin_projections(problem_set: QNProblemSet) -> QNProblemSet:
    facts = problem_set.initial_facts
    new_states = {
        edge_id: {
            **prop_map,
            EdgeQuantumNumbers.spin_projection: spin_projections[
                PDG.find(int(prop_map[EdgeQuantumNumbers.pid])).name
            ],
        }
        for edge_id, prop_map in facts.states.items()
    }
    new_facts = MutableTransition(facts.topology, new_states, dict(facts.interactions))
    settings = problem_set.solving_settings
    new_edge_settings = {}
    for edge_id, edge_settings in settings.states.items():
        if edge_id not in facts.topology.intermediate_edge_ids:
            new_edge_settings[edge_id] = edge_settings
            continue
        max_spin = max(edge_settings.qn_domains[EdgeQuantumNumbers.spin_magnitude])
        projection_domain = [
            Fraction(x, 2) for x in range(-int(2 * max_spin), int(2 * max_spin) + 1)
        ]
        new_edge_settings[edge_id] = attrs.evolve(
            edge_settings,
            conservation_rules={
                **edge_settings.conservation_rules,
                spin_validity: EDGE_RULE_PRIORITIES[spin_validity],
            },
            qn_domains={
                **edge_settings.qn_domains,
                EdgeQuantumNumbers.spin_projection: projection_domain,
            },
        )
    new_settings = MutableTransition(
        settings.topology, new_edge_settings, dict(settings.interactions)
    )
    return QNProblemSet(initial_facts=new_facts, solving_settings=new_settings)


collection.problem_sets = {
    strength: [add_spin_projections(problem_set) for problem_set in problem_sets]
    for strength, problem_sets in collection.problem_sets.items()
}

The solver now propagates the projections just like any other quantum number:

In [ ]:
qn_transitions = find_qn_transitions(collection)
len(qn_transitions)

Every transition carries a spin projection for every state, but nothing constrains the projections yet. All combinations of the photon helicity $\lambda_\gamma$ and the $f_2(1270)$ helicity $\lambda_{f_2}$ appear:

In [ ]:
def get_helicity_combinations(transitions) -> dict[int, list[int]]:
    combinations = defaultdict(set)
    for transition in transitions:
        topology = transition.topology
        resonance_edge = next(iter(topology.intermediate_edge_ids))
        gamma_edge = next(
            i
            for i in topology.outgoing_edge_ids
            if transition.states[i][EdgeQuantumNumbers.pid] == 22
        )
        lambda_gamma = transition.states[gamma_edge][EdgeQuantumNumbers.spin_projection]
        lambda_f2 = transition.states[resonance_edge][
            EdgeQuantumNumbers.spin_projection
        ]
        combinations[int(lambda_gamma)].add(int(lambda_f2))
    return {k: sorted(v) for k, v in sorted(combinations.items())}

In [ ]:
get_helicity_combinations(qn_transitions)

## Adding a conservation rule

The projections become meaningful once a **rule** constrains them. {func}`.helicity_conservation` checks $|\lambda_2-\lambda_3| \leq S_1$ at each decay node $1 \to 2\,3$. Adding it to the conservation rules of the interaction nodes works the same way as adding {func}`.spin_validity` to the edges:

In [ ]:
def add_helicity_conservation(problem_set: QNProblemSet) -> QNProblemSet:
    settings = problem_set.solving_settings
    new_node_settings = {
        node_id: attrs.evolve(
            node_settings,
            conservation_rules={
                **node_settings.conservation_rules,
                helicity_conservation: CONSERVATION_LAW_PRIORITIES[
                    helicity_conservation
                ],
            },
        )
        for node_id, node_settings in settings.interactions.items()
    }
    new_settings = MutableTransition(
        settings.topology, dict(settings.states), new_node_settings
    )
    return QNProblemSet(
        initial_facts=problem_set.initial_facts, solving_settings=new_settings
    )


collection.problem_sets = {
    strength: [add_helicity_conservation(problem_set) for problem_set in problem_sets]
    for strength, problem_sets in collection.problem_sets.items()
}
qn_transitions = find_qn_transitions(collection)
len(qn_transitions)

At the production node, the rule requires $|\lambda_\gamma - \lambda_{f_2}| \leq S_{J/\psi} = 1$, which prunes the helicity combinations to:

In [ ]:
get_helicity_combinations(qn_transitions)

The resulting transitions render like any other quantum-number transition, now with the solved spin projections included:

In [ ]:
dot = qrules.io.asdot(qn_transitions[0], render_node=True)
graphviz.Source(dot)

:::{tip}
The same mechanism works for any quantum number in {class}`.EdgeQuantumNumbers` and {class}`.NodeQuantumNumbers` and for any rule that consumes them — including rules of your own, as described in {doc}`/usage/conservation`.
:::